# Predição de Preços de Ações usando LSTM

## Introdução

Este notebook implementa um modelo de **Redes Neurais LSTM** para prever o preço de fechamento de ações da B3. 

Baseado no modelo descrito no TCC *"Predição de Preços do Mercado Financeiro utilizando Redes Neurais LSTM"*, seguimos as seguintes etapas:

1. **Carregar e Pré-processar os Dados**
2. **Criar e Treinar o Modelo LSTM**
3. **Avaliar o Modelo com Métricas (MSE e RMSE)**
4. **Visualizar os Resultados**



In [2]:
# Importação de Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.losses import MeanSquaredError
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Carregar o dataset
file_path = "../datasets/b3_dados/processed/acoes_concat.csv"  # Atualize com o caminho correto
df = pd.read_csv(file_path)

# Converter a coluna de data para datetime
df['Date'] = pd.to_datetime(df['Date'])

# Exibir as primeiras linhas
df.head()


,Date,ITUB4,BBAS3,CYRE3,TEND3,DIRR3,ELET3,EQTL3,CMIG4,PETR3,VALE3,BRAP3
0,2010-01-04,10.562274,5.562300,14.371702,5.135384,5.177327,20.123955,2.727207,1.917287,11.676325,23.756481,3.748754
1,2010-01-05,10.630730,5.506486,14.235113,5.089202,5.154717,19.958874,2.748718,1.946624,11.606239,23.977951,3.767592
2,2010-01-06,10.538581,5.513929,14.252928,5.089202,5.109501,19.703260,2.719525,1.909956,11.721180,24.485470,3.805269
3,2010-01-07,10.430645,5.515787,13.979749,5.061493,5.154717,20.161232,2.688796,1.891619,11.651093,24.586971,3.852364
4,2010-01-08,10.272685,5.547416,13.955992,4.950658,5.163762,20.502045,2.728743,1.889784,11.564186,24.826887,3.899458


## Pré-processamento dos Dados

In [4]:
# Normalizar os dados (exceto a coluna Date)
scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled = df.copy()
df_scaled.iloc[:, 1:] = scaler.fit_transform(df.iloc[:, 1:])

# Criar janelas de entrada (3 dias)
def create_sequences(data, seq_length=3):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Aplicar a criação de janelas para cada ação
seq_length = 3
data_sequences = {}
for stock in df.columns[1:]:  
    X, y = create_sequences(df_scaled[stock].values, seq_length)
    data_sequences[stock] = (X, y)

# Divisão dos dados (60% treino, 10% validação, 30% teste)
train_size = int(0.6 * len(df_scaled))
val_size = int(0.1 * len(df_scaled))

split_data = {}
for stock, (X, y) in data_sequences.items():
    split_data[stock] = {
        "train": (X[:train_size], y[:train_size]),
        "val": (X[train_size:train_size+val_size], y[train_size:train_size+val_size]),
        "test": (X[train_size+val_size:], y[train_size+val_size:])
    }


## Construção do Modelo LSTM

In [ ]:
# Criar o Modelo LSTM
def create_lstm_model(input_shape):
    model = Sequential([
        LSTM(105, return_sequences=False, input_shape=input_shape),
        Dropout(0.2),
        Dense(7, activation="relu"),
        Dense(1, activation="linear")
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

# Criar e treinar o modelo para cada ação
models = {}
for stock in split_data.keys():
    X_train, y_train = split_data[stock]["train"]
    input_shape = (X_train.shape[1], 1)
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

    model = create_lstm_model(input_shape)
    model.fit(X_train, y_train, epochs=100, batch_size=1, verbose=1)
    
    # Salvar modelo treinado
    model.save(f"models/modelo_lstm_{stock}.h5")
    models[stock] = model


## Visualização dos Resultados

In [ ]:
# Escolher ação para visualizar
stock = "ITUB4"  # Troque pelo código desejado

# Criar instância de MSE para carregar o modelo corretamente
mse = MeanSquaredError()

# Carregar modelo salvo
modelo = load_model(f"models/modelo_lstm_{stock}.h5", compile=False)

# Carregar dados de teste
X_test, y_test = split_data[stock]["test"]
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Fazer previsões
y_pred = modelo.predict(X_test)

# Criar novo scaler para reverter normalização
scaler_stock = MinMaxScaler(feature_range=(0, 1))
df_stock = df[[stock]]
df_stock_scaled = scaler_stock.fit_transform(df_stock)

# Reverter normalização
y_test_real = scaler_stock.inverse_transform(y_test.reshape(-1, 1))
y_pred_real = scaler_stock.inverse_transform(y_pred)

# Gerar gráfico comparativo
plt.figure(figsize=(12, 6))
plt.plot(df['Date'][-len(y_test):], y_test_real, label="Valor Real", color='blue')
plt.plot(df['Date'][-len(y_pred):], y_pred_real, label="Valor Previsto", color='red', linestyle="dashed")
plt.xlabel("Data")
plt.ylabel("Preço de Fechamento")
plt.title(f"Comparação de Previsões LSTM para {stock}")
plt.legend()
plt.xticks(rotation=45)
plt.grid()
plt.show()


## Avaliação do Modelo

In [ ]:
# Avaliação do Modelo

# Calcular MSE
mse_value = mean_squared_error(y_test_real, y_pred_real)

# Calcular RMSE
rmse_value = np.sqrt(mse_value)

print(f"MSE: {mse_value:.5f}")
print(f"RMSE: {rmse_value:.5f}")
